In [1]:
# ============================================================
# Energy Efficiency Analysis of Autonomous Vehicles
# Machine Learning Model using Linear Regression
# ============================================================
# Libraries used:
#   pandas      - for data handling
#   matplotlib  - for visualization
#   scikit-learn - for ML model and evaluation
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# STEP 1: GENERATE SYNTHETIC DATASET
# ============================================================

print("=" * 60)
print("  ENERGY EFFICIENCY ANALYSIS OF AUTONOMOUS VEHICLES")
print("=" * 60)
print()

# Set a random seed so results are consistent every run
np.random.seed(42)
n_samples = 150  # 150 rows of data

# Generate realistic vehicle parameter data
speed        = np.random.uniform(20, 120, n_samples)        # km/h
acceleration = np.random.uniform(0.5, 4.0, n_samples)       # m/s²
distance     = np.random.uniform(5, 100, n_samples)         # km

# Energy consumed depends on speed, acceleration, and distance
# Formula inspired by real EV physics:
#   - Higher speed → more air resistance → more energy
#   - Higher acceleration → more power needed
#   - Longer distance → more total energy used
energy_consumed = (
    0.15 * speed +
    2.5  * acceleration +
    0.8  * distance +
    0.003 * speed**2 +            # aerodynamic drag increases with speed squared
    np.random.normal(0, 3, n_samples)  # small random noise
)

# Put it all in a DataFrame (like a spreadsheet)
df = pd.DataFrame({
    'Speed':           np.round(speed, 2),
    'Acceleration':    np.round(acceleration, 2),
    'Distance':        np.round(distance, 2),
    'Energy_Consumed': np.round(energy_consumed, 2)
})

# Save the dataset to a CSV file
df.to_csv('ev_dataset.csv', index=False)
print("✅ Synthetic dataset generated and saved as 'ev_dataset.csv'")
print()

# ============================================================
# STEP 2: DATA LOADING
# ============================================================

print("--- STEP 1: DATA LOADING ---")
data = pd.read_csv('ev_dataset.csv')
print(f"Dataset Shape: {data.shape[0]} rows x {data.shape[1]} columns")
print()
print("First 5 rows of the dataset:")
print(data.head())
print()

# ============================================================
# STEP 3: DATA PREPROCESSING
# ============================================================

print("--- STEP 2: DATA PREPROCESSING ---")

# Check for missing values
print("Missing values in each column:")
print(data.isnull().sum())
print()

# Handle missing values if any (fill with column mean)
if data.isnull().sum().any():
    data = data.fillna(data.mean())
    print("Missing values filled with column mean.")
else:
    print("✅ No missing values found. Dataset is clean!")
print()

# Summary statistics — min, max, average, etc.
print("Summary Statistics:")
print(data.describe().round(2))
print()

# Separate features (X) and target (y)
X = data[['Speed', 'Acceleration', 'Distance']]  # Input features
y = data['Energy_Consumed']                        # Target variable to predict

print("Features (X): Speed, Acceleration, Distance")
print("Target  (y): Energy_Consumed")
print()

# ============================================================
# STEP 4: TRAIN-TEST SPLIT (80% train, 20% test)
# ============================================================

print("--- STEP 3: TRAIN-TEST SPLIT ---")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Training samples : {len(X_train)}")
print(f"Testing  samples : {len(X_test)}")
print()

# ============================================================
# STEP 5: MODEL BUILDING — LINEAR REGRESSION
# ============================================================

print("--- STEP 4: MODEL BUILDING ---")
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
print("✅ Linear Regression model trained successfully!")
print()

# Show what coefficients the model learned
print("Model Coefficients (what the model learned):")
for feature, coef in zip(X.columns, lr_model.coef_):
    print(f"  {feature:15s}: {coef:.4f}")
print(f"  {'Intercept':15s}: {lr_model.intercept_:.4f}")
print()

# Also train a Random Forest for comparison
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
print("✅ Random Forest model also trained for comparison!")
print()

# ============================================================
# STEP 6: PREDICTIONS
# ============================================================

print("--- STEP 5: PREDICTIONS ---")
y_pred_lr = lr_model.predict(X_test)   # Linear Regression predictions
y_pred_rf = rf_model.predict(X_test)   # Random Forest predictions
print("Predictions made on test data.")
print()

# Show a few actual vs predicted values
comparison = pd.DataFrame({
    'Actual': y_test.values[:10],
    'LR Predicted': np.round(y_pred_lr[:10], 2),
    'RF Predicted': np.round(y_pred_rf[:10], 2)
})
print("Sample Actual vs Predicted (first 10 rows):")
print(comparison.to_string(index=False))
print()

# ============================================================
# STEP 7: MODEL EVALUATION
# ============================================================

print("--- STEP 6: MODEL EVALUATION ---")

# Calculate RMSE and R² for Linear Regression
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr   = r2_score(y_test, y_pred_lr)

# Calculate RMSE and R² for Random Forest
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf   = r2_score(y_test, y_pred_rf)

print(f"{'Model':<22} {'RMSE':>10} {'R² Score':>10}")
print("-" * 44)
print(f"{'Linear Regression':<22} {rmse_lr:>10.4f} {r2_lr:>10.4f}")
print(f"{'Random Forest':<22} {rmse_rf:>10.4f} {r2_rf:>10.4f}")
print()

# Simple explanation of results
print("📊 RESULT INTERPRETATION:")
print(f"  • Linear Regression R² = {r2_lr:.4f}")
print(f"    → The model explains {r2_lr*100:.1f}% of the variation in energy consumption.")
print(f"  • RMSE = {rmse_lr:.4f} kWh")
print(f"    → On average, predictions are off by ±{rmse_lr:.2f} kWh.")
if r2_lr >= 0.85:
    print("  ✅ Excellent fit! The model performs very well.")
elif r2_lr >= 0.70:
    print("  ✅ Good fit. The model captures most patterns in the data.")
else:
    print("  ⚠️  Moderate fit. Consider feature engineering or more data.")
print()

# ============================================================
# STEP 8: VISUALIZATIONS
# ============================================================

print("--- STEP 7: GENERATING VISUALIZATIONS ---")

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('Energy Efficiency Analysis of Autonomous Vehicles',
             fontsize=16, fontweight='bold', y=1.01)

# --- Plot 1: Actual vs Predicted (Linear Regression) ---
ax1 = axes[0, 0]
ax1.scatter(y_test, y_pred_lr, color='steelblue', alpha=0.7, edgecolors='white', s=80)
min_val = min(y_test.min(), y_pred_lr.min())
max_val = max(y_test.max(), y_pred_lr.max())
ax1.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
ax1.set_xlabel('Actual Energy Consumed (kWh)', fontsize=11)
ax1.set_ylabel('Predicted Energy Consumed (kWh)', fontsize=11)
ax1.set_title(f'Actual vs Predicted\n(Linear Regression, R²={r2_lr:.3f})', fontsize=12)
ax1.legend()
ax1.grid(True, alpha=0.3)

# --- Plot 2: Speed vs Energy Consumption ---
ax2 = axes[0, 1]
scatter = ax2.scatter(data['Speed'], data['Energy_Consumed'],
                      c=data['Distance'], cmap='viridis', alpha=0.7, s=60)
plt.colorbar(scatter, ax=ax2, label='Distance (km)')
ax2.set_xlabel('Speed (km/h)', fontsize=11)
ax2.set_ylabel('Energy Consumed (kWh)', fontsize=11)
ax2.set_title('Speed vs Energy Consumption\n(Color = Distance)', fontsize=12)
ax2.grid(True, alpha=0.3)

# --- Plot 3: Distance vs Energy Consumption ---
ax3 = axes[1, 0]
scatter2 = ax3.scatter(data['Distance'], data['Energy_Consumed'],
                       c=data['Speed'], cmap='plasma', alpha=0.7, s=60)
plt.colorbar(scatter2, ax=ax3, label='Speed (km/h)')
ax3.set_xlabel('Distance (km)', fontsize=11)
ax3.set_ylabel('Energy Consumed (kWh)', fontsize=11)
ax3.set_title('Distance vs Energy Consumption\n(Color = Speed)', fontsize=12)
ax3.grid(True, alpha=0.3)

# --- Plot 4: Model Comparison Bar Chart ---
ax4 = axes[1, 1]
models = ['Linear Regression', 'Random Forest']
r2_scores = [r2_lr, r2_rf]
colors = ['steelblue', 'coral']
bars = ax4.bar(models, r2_scores, color=colors, edgecolor='white', width=0.5)
ax4.set_ylim(0, 1.05)
ax4.set_ylabel('R² Score', fontsize=11)
ax4.set_title('Model Comparison\n(R² Score — Higher is Better)', fontsize=12)
for bar, score in zip(bars, r2_scores):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{score:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax4.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('ev_analysis_plots.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Visualizations saved as 'ev_analysis_plots.png'")
print()

# ============================================================
# STEP 9: OUTPUT EXPLANATION
# ============================================================

print("=" * 60)
print("  WHAT THE MODEL LEARNED")
print("=" * 60)
print()
print("The Linear Regression model found these relationships:")
print()
print("  1. SPEED:")
print("     → For every 1 km/h increase in speed,")
print(f"       energy consumption increases by ~{lr_model.coef_[0]:.2f} kWh")
print("     → Higher speeds require more power to overcome air resistance.")
print()
print("  2. ACCELERATION:")
print("     → For every 1 m/s² increase in acceleration,")
print(f"       energy consumption increases by ~{lr_model.coef_[1]:.2f} kWh")
print("     → Aggressive acceleration burns significantly more energy.")
print()
print("  3. DISTANCE:")
print("     → For every 1 km increase in distance,")
print(f"       energy consumption increases by ~{lr_model.coef_[2]:.2f} kWh")
print("     → Longer trips naturally consume more energy.")
print()
print("KEY TAKEAWAY:")
print("  Driving at moderate, steady speeds with gentle acceleration")
print("  is the most energy-efficient strategy for autonomous vehicles.")
print()
print("=" * 60)
print("  ANALYSIS COMPLETE!")
print("=" * 60)


  ENERGY EFFICIENCY ANALYSIS OF AUTONOMOUS VEHICLES

✅ Synthetic dataset generated and saved as 'ev_dataset.csv'

--- STEP 1: DATA LOADING ---
Dataset Shape: 150 rows x 4 columns

First 5 rows of the dataset:
    Speed  Acceleration  Distance  Energy_Consumed
0   57.45          3.68      9.91            36.58
1  115.07          1.34     55.48           109.14
2   93.20          1.01     56.36            90.22
3   79.87          2.21     65.56            88.61
4   35.60          3.95     73.98            78.14

--- STEP 2: DATA PREPROCESSING ---
Missing values in each column:
Speed              0
Acceleration       0
Distance           0
Energy_Consumed    0
dtype: int64

✅ No missing values found. Dataset is clean!

Summary Statistics:
        Speed  Acceleration  Distance  Energy_Consumed
count  150.00        150.00    150.00           150.00
mean    67.29          2.31     52.13            73.85
std     29.65          1.02     28.63            26.51
min     20.55          0.52      6